# GLI3

Cohort: `GLI3`, Phenopacket Store v0.1.27 (n = 81).

- Greig cephalopolysyndactyly syndrome (GCPS), [OMIM:175700](https://omim.org/entry/175700), n = 50
- Pallister-Hall syndrome (PHS), [OMIM:146510](https://omim.org/entry/146510), n = 21
- Postaxial polydactyly, types A1 and B, [OMIM:174200](https://omim.org/entry/174200), n = 10

GCPS and PHS are distinct entities caused by variants in the same gene, and the split is
predictable from variant type and position: haploinsufficiency causes GCPS, whereas
truncating variants in the middle third of the gene, 3' of the zinc-finger domain, cause
PHS (Johnston et al. 2005, PMID 15739154; Demurger et al. 2015, Eur J Hum Genet).

In [1]:
from pathlib import Path

import pandas as pd
from ppktstore.registry import configure_phenopacket_registry

from phenosign import (
    PhenotypeDatasetBuilder,
    HPOCorrelationAnalyzer,
    SynergyAnalyzer,
    has_disease,
)

OUTDIR = Path("../case_studies/GLI3")
OUTDIR.mkdir(parents=True, exist_ok=True)

COLS = ["HPO_A_label", "HPO_B_label", "synergy", "adj_p_value", "n_individuals", "n_pmids"]

In [2]:
registry = configure_phenopacket_registry()
with registry.open_phenopacket_store(release="0.1.27") as ps:
    phenopackets = list(ps.iter_cohort_phenopackets("GLI3"))

print(f"Loaded {len(phenopackets)} phenopackets")

Loaded 81 phenopackets


In [3]:
dataset = PhenotypeDatasetBuilder(phenopackets).build(build_gpsea_cohort=False)

In [4]:
diseases_df, sex_df, genes_df, _ = dataset.describe_conditions()
diseases_df

,label,n_individuals
disease_id,,
OMIM:175700,Greig cephalopolysyndactyly syndrome,50 (61.7%)
OMIM:146510,Pallister-Hall syndrome,21 (25.9%)
OMIM:174200,"Polydactyly, postaxial, types A1 and B",10 (12.3%)


## Pairwise correlation

In [5]:
analyzer = HPOCorrelationAnalyzer(dataset=dataset)
results = analyzer.compute_correlation_matrix()

t = results.results_table
print(f"tested pairs: {len(t)} | significant: {int((t['adj_p_value'] < 0.05).sum())}")
t.head()

Calculating pairwise correlation: 100%|██████████| 123/123 [00:01<00:00, 79.41it/s]

tested pairs: 82 | significant: 20


,HPO_A,HPO_A_label,HPO_B,HPO_B_label,correlation,p_value,adj_p_value,n(A:E/B:E),n(A:E/B:O),n(A:O/B:E),n(A:O/B:O),n_individuals,n_pmids,pmids
77,HP:0006042,Y-shaped metacarpals,HP:0010567,Y-shaped metatarsals,1.000000,2.254063e-15,1.848331e-13,53,0,0,15,68,1,24736735
66,HP:0001841,Preaxial foot polydactyly,HP:0010567,Y-shaped metatarsals,-0.581203,6.774364e-07,2.777489e-05,16,15,37,0,68,1,24736735
80,HP:0010554,Cutaneous finger syndactyly,HP:0010621,Cutaneous syndactyly of toes,1.000000,4.343483e-06,1.187219e-04,20,0,0,6,26,3,18000979;22428873;24736735
74,HP:0002164,Nail dysplasia,HP:0006042,Y-shaped metacarpals,0.680998,9.722603e-06,1.993134e-04,60,4,1,6,71,3,18000979;22428873;24736735
63,HP:0001841,Preaxial foot polydactyly,HP:0006042,Y-shaped metacarpals,-0.463537,1.666619e-05,2.733255e-04,26,15,37,0,78,3,18000979;22428873;24736735


In [6]:
results.save_correlation_results(
    output_file=str(OUTDIR / "correlation_results_GLI3.csv")
)

In [7]:
sig = results.results_table.query("adj_p_value < 0.05")
sig = sig.reindex(sig["correlation"].abs().sort_values(ascending=False).index)
sig[["HPO_A_label", "HPO_B_label", "correlation", "adj_p_value", "n_individuals", "n_pmids"]]

,HPO_A_label,HPO_B_label,correlation,adj_p_value,n_individuals,n_pmids
77,Y-shaped metacarpals,Y-shaped metatarsals,1.000000,1.848331e-13,68,1
80,Cutaneous finger syndactyly,Cutaneous syndactyly of toes,1.000000,1.187219e-04,26,3
48,Intellectual disability,Anal atresia,0.738549,2.097187e-02,25,3
74,Nail dysplasia,Y-shaped metacarpals,0.680998,1.993134e-04,71,3
75,Nail dysplasia,Y-shaped metatarsals,0.674156,3.392618e-04,61,1
73,Anal atresia,Postaxial polydactyly,-0.595581,4.099185e-03,57,3
66,Preaxial foot polydactyly,Y-shaped metatarsals,-0.581203,2.777489e-05,68,1
70,Anal atresia,Nail dysplasia,0.558144,4.099185e-03,73,3
52,Growth delay,Anal atresia,0.542105,4.432501e-02,29,3
27,Postaxial hand polydactyly,Postaxial foot polydactyly,0.524873,9.455051e-04,54,1


In [8]:
results.plot_correlation_heatmap_with_significance(
    title_name="Cohort GLI3",
)

## Synergy: Pallister-Hall vs other GLI3

In [9]:
condition = dataset.get_condition(
    has_disease("OMIM:146510"),
    name="Pallister-Hall vs other GLI3",
)

condition.value_counts(dropna=False)

Pallister-Hall vs other GLI3
0.0    60
1.0    21
Name: count, dtype: int64

In [10]:
# lowered from the default of 30, the cohort splits 21/60
synergy_analyzer = SynergyAnalyzer(
    dataset=dataset,
    min_individuals_for_synergy_calculation=20,
)
synergy_phs = synergy_analyzer.compute_synergy_matrix(condition=condition)

st = synergy_phs.results_table
print(
    f"rows={len(st)} | synergy {st['synergy'].min():.4f} to {st['synergy'].max():.4f}"
    f" | positive={(st['synergy'] > 0).sum()} negative={(st['synergy'] < 0).sum()}"
)
st.head()

Calculating pairwise synergy: 100%|██████████| 123/123 [00:00<00:00, 5778.18it/s]


rows=42 | synergy -0.3059 to 0.0779 | positive=5 negative=37


,HPO_A,HPO_A_label,HPO_B,HPO_B_label,synergy,p_value,adj_p_value,n(A:E/B:E)_y0,n(A:E/B:O)_y0,n(A:O/B:E)_y0,n(A:O/B:O)_y0,N_y0,n(A:E/B:E)_y1,n(A:E/B:O)_y1,n(A:O/B:E)_y1,n(A:O/B:O)_y1,N_y1,n_individuals,n_pmids,pmids
37,HP:0002164,Nail dysplasia,HP:0006042,Y-shaped metacarpals,-0.222973,0.0001,0.000467,60,0,0,0,60,0,4,1,6,11,71,3,18000979;22428873;24736735
31,HP:0001841,Preaxial foot polydactyly,HP:0010567,Y-shaped metatarsals,-0.305922,0.0001,0.000467,13,0,37,0,50,3,15,0,0,18,68,1,24736735
30,HP:0001841,Preaxial foot polydactyly,HP:0006042,Y-shaped metacarpals,-0.208328,0.0001,0.000467,23,0,37,0,60,3,15,0,0,18,78,3,18000979;22428873;24736735
29,HP:0001841,Preaxial foot polydactyly,HP:0002164,Nail dysplasia,-0.138637,0.0001,0.000467,23,0,37,0,60,4,9,0,0,13,73,3,18000979;22428873;24736735
28,HP:0001841,Preaxial foot polydactyly,HP:0002023,Anal atresia,-0.119325,0.0001,0.000467,23,0,37,0,60,11,10,0,0,21,81,3,18000979;22428873;24736735


In [11]:
# save_synergy_results() drops everything below synergy >= 0.01,
# so keep the unfiltered table for the dissynergy side
synergy_phs.results_table.to_csv(
    OUTDIR / "synergy_full_GLI3_OMIM_146510.csv", index=False
)
synergy_phs.save_synergy_results(
    output_file=str(OUTDIR / "synergy_matrix_GLI3_OMIM_146510.csv")
)

In [12]:
synergy_phs.plot_synergy_heatmap()

In [13]:
print("--- top synergy ---")
display(st.nlargest(10, "synergy")[COLS])
print("--- top dissynergy ---")
display(st.nsmallest(10, "synergy")[COLS])

--- top synergy ---


,HPO_A_label,HPO_B_label,synergy,adj_p_value,n_individuals,n_pmids
2,Postaxial hand polydactyly,Preaxial foot polydactyly,0.077857,0.013359,64,3
17,Intellectual disability,Y-shaped metacarpals,0.046846,0.207252,23,3
23,Postaxial foot polydactyly,Preaxial foot polydactyly,0.030073,0.067066,54,1
41,Y-shaped metatarsals,Postaxial polydactyly,0.022780,0.186826,44,1
40,Y-shaped metacarpals,Postaxial polydactyly,0.014101,0.177527,54,3
13,Preaxial hand polydactyly,Postaxial polydactyly,-0.003731,0.050949,57,3
0,Postaxial hand polydactyly,Preaxial hand polydactyly,-0.006363,0.350772,64,3
7,Preaxial hand polydactyly,Postaxial foot polydactyly,-0.006491,0.687016,54,1
9,Preaxial hand polydactyly,Anal atresia,-0.007190,0.004941,81,3
10,Preaxial hand polydactyly,Nail dysplasia,-0.007937,0.006039,73,3


--- top dissynergy ---


,HPO_A_label,HPO_B_label,synergy,adj_p_value,n_individuals,n_pmids
31,Preaxial foot polydactyly,Y-shaped metatarsals,-0.305922,0.000467,68,1
38,Nail dysplasia,Y-shaped metatarsals,-0.238793,0.000467,61,1
37,Nail dysplasia,Y-shaped metacarpals,-0.222973,0.000467,71,3
30,Preaxial foot polydactyly,Y-shaped metacarpals,-0.208328,0.000467,78,3
19,Growth delay,Anal atresia,-0.184158,0.050949,29,3
20,Growth delay,Nail dysplasia,-0.170782,0.128368,21,3
21,Growth delay,Y-shaped metacarpals,-0.151533,0.067066,28,3
33,Anal atresia,Nail dysplasia,-0.141379,0.000467,73,3
29,Preaxial foot polydactyly,Nail dysplasia,-0.138637,0.000467,73,3
28,Preaxial foot polydactyly,Anal atresia,-0.119325,0.000467,81,3


## Reverse contrast: GCPS vs other GLI3

Better balanced split (50/31).

In [14]:
condition_gcps = dataset.get_condition(
    has_disease("OMIM:175700"),
    name="GCPS vs other GLI3",
)
synergy_gcps = synergy_analyzer.compute_synergy_matrix(condition=condition_gcps)

sg = synergy_gcps.results_table
sg.to_csv(OUTDIR / "synergy_full_GLI3_OMIM_175700.csv", index=False)
synergy_gcps.save_synergy_results(
    output_file=str(OUTDIR / "synergy_matrix_GLI3_OMIM_175700.csv")
)

print(
    f"rows={len(sg)} | synergy {sg['synergy'].min():.4f} to {sg['synergy'].max():.4f}"
)

Calculating pairwise synergy: 100%|██████████| 123/123 [00:22<00:00,  5.48it/s]


rows=68 | synergy -0.3059 to 0.1103


In [15]:
synergy_gcps.plot_synergy_heatmap()

In [16]:
print("--- top synergy ---")
display(sg.nlargest(10, "synergy")[COLS])
print("--- top dissynergy ---")
display(sg.nsmallest(10, "synergy")[COLS])

--- top synergy ---


,HPO_A_label,HPO_B_label,synergy,adj_p_value,n_individuals,n_pmids
26,Postaxial hand polydactyly,Anal atresia,0.110292,0.000618,64,3
27,Postaxial hand polydactyly,Nail dysplasia,0.084192,0.000971,62,3
56,Preaxial foot polydactyly,Broad thumb,0.076141,0.032359,21,2
19,Syndactyly,Preaxial foot polydactyly,0.075213,0.012086,54,2
66,Broad hallux,Broad thumb,0.058614,0.032719,21,2
17,Syndactyly,Postaxial hand polydactyly,0.044966,0.053375,54,2
2,Macrocephaly,Postaxial hand polydactyly,0.031149,0.144212,57,3
44,Postaxial foot polydactyly,Preaxial foot polydactyly,0.030812,0.058453,54,1
28,Postaxial hand polydactyly,Y-shaped metacarpals,0.028693,0.007615,64,3
67,Y-shaped metatarsals,Postaxial polydactyly,0.022780,0.211737,44,1


--- top dissynergy ---


,HPO_A_label,HPO_B_label,synergy,adj_p_value,n_individuals,n_pmids
54,Preaxial foot polydactyly,Y-shaped metatarsals,-0.305922,0.000618,68,1
5,Macrocephaly,Broad hallux,-0.260214,0.102550,20,2
63,Nail dysplasia,Y-shaped metatarsals,-0.238793,0.000618,61,1
34,Postaxial hand polydactyly,Preaxial polydactyly,-0.230887,0.000618,47,3
9,Macrocephaly,Preaxial polydactyly,-0.222011,0.000618,44,3
7,Macrocephaly,Cutaneous syndactyly of toes,-0.213165,0.003777,25,3
6,Macrocephaly,Cutaneous finger syndactyly,-0.213165,0.003777,25,3
51,Preaxial foot polydactyly,Y-shaped metacarpals,-0.208254,0.000618,78,3
62,Nail dysplasia,Y-shaped metacarpals,-0.171443,0.000618,71,3
16,Hypertelorism,Preaxial polydactyly,-0.163011,0.000618,39,3
